# 게임 모집단 DB 추출 및 전처리

PostgreSQL DB에서 Steam 인디게임 데이터를 추출하여 `data/raw/`에 원본 CSV로 저장하고,
분석 목적에 맞게 유료 인디게임 모집단을 전처리하여 `data/preprocessed/`에 저장한다.

> **실행 순서:** 이 노트북 → `01_stratified_sampling.ipynb` → 수집 스크립트 → `02_preprocess_reviews.ipynb`

## DB 추출 대상

| 테이블 | 설명 | 저장 파일 |
|---|---|---|
| `steam_indie_tags` | SteamSpy 태그 데이터 | `data/raw/steam_indie_tags.csv` |
| `steam_app_details` | 게임 상세 정보 (Indie 장르만) | `data/raw/steam_app_details.csv` |
| `steamspy_indie_games` | SteamSpy 수집 인디게임 전체 데이터 | `data/raw/steamspy_indie_games.csv` |

## 전처리 결과

| 대상 | 설명 | 저장 파일 |
|---|---|---|
| `steam_indie_games` | 초기 반응 그룹: 리뷰 10개 이상, EA/F2P/price=0/부적합 장르/성인 태그 제외 | `data/preprocessed/steam_indie_games.csv` |
| `steam_indie_games_silence` | 침묵 그룹: 리뷰 0~9개, 위와 동일한 기간·EA/F2P/price=0·장르·성인 태그 필터 적용 | `data/preprocessed/steam_indie_games_silence.csv` |

## 주요 전처리 원칙

- `data/raw/`는 원본 보존용이며 수정하지 않는다.
- 가격 전략 분석의 일관성을 위해 `Free To Play` 장르와 `price_spy <= 0` 게임은 메인/침묵 모집단에서 제외한다.
- 출시일은 SteamSpy 원본 `release_date`를 우선 보존하고, 결측일 때만 Store `release_date`로 보완한다.
- 침묵 그룹은 현재 태그 미수집 게임이 포함될 수 있으므로 `tags` 병합은 left join으로 수행한다.

## 라이브러리 임포트 및 데이터 로드

In [1]:
import ast
import json as _json
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

warnings.filterwarnings('ignore')

# VS Code 노트북: __vsc_ipynb_file__ 로 프로젝트 루트 계산
# 노트북 위치: <project>/src/notebooks/jin/ → parents[2] = project root
if '__vsc_ipynb_file__' in globals():
    PROJECT_ROOT = Path(globals()['__vsc_ipynb_file__']).parents[2]
else:
    import subprocess
    _out = subprocess.run(
        ['git', 'rev-parse', '--show-toplevel'],
        capture_output=True, text=True, cwd=Path.home()
    )
    PROJECT_ROOT = Path(_out.stdout.strip())

RAW_DIR          = PROJECT_ROOT / 'data' / 'raw'
PREPROCESSED_DIR = PROJECT_ROOT / 'data' / 'preprocessed'
RAW_DIR.mkdir(parents=True, exist_ok=True)
PREPROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f'PROJECT_ROOT : {PROJECT_ROOT}')
print(f'RAW_DIR      : {RAW_DIR}')
print(f'PREPROCESSED : {PREPROCESSED_DIR}')

PROJECT_ROOT : /Users/jin/Develop/codingclub/game-analysis
RAW_DIR      : /Users/jin/Develop/codingclub/game-analysis/data/raw
PREPROCESSED : /Users/jin/Develop/codingclub/game-analysis/data/preprocessed


## 2. steam_app_details

Steam Store API로 수집한 게임 상세 정보 (설명, 장르, 카테고리 등 전체 필드). `steamspy_indie_games`와 inner join하여 수집 대상 게임만 조회한다.

In [2]:
df_app_details = pd.read_csv(RAW_DIR / 'steam_app_details.csv')

print(f'steam_app_details: {len(df_app_details):,}행')
print(f'컬럼: {df_app_details.columns.tolist()}')
df_app_details.head(3)

steam_app_details: 58,453행
컬럼: ['appid', 'name', 'type', 'is_free', 'controller_support', 'short_description', 'supported_languages', 'developers', 'publishers', 'genres', 'categories', 'coming_soon', 'release_date', 'currency', 'initial', 'final', 'discount_percent', 'initial_formatted', 'final_formatted', 'windows', 'mac', 'linux', 'recommendations_total', 'metacritic_score', 'metacritic_url', 'achievements_total', 'header_image', 'website']


,appid,name,type,is_free,controller_support,short_description,supported_languages,developers,publishers,genres,...,final_formatted,windows,mac,linux,recommendations_total,metacritic_score,metacritic_url,achievements_total,header_image,website
0,1002,Rag Doll Kung Fu,game,False,NaN,A piece of Steam history - THE FIRST EVER NON ...,English,Mark Healey,Mark Healey,Indie,...,"₩ 1,100",True,False,False,NaN,69.0,https://www.metacritic.com/game/pc/rag-doll-ku...,NaN,https://shared.akamai.steamstatic.com/store_it...,http://www.ragdollkungfu.com/
1,1500,Darwinia,game,False,full,"Darwinia blends real-time strategy, action, an...","English, German, French, Italian, Spanish - Spain",Introversion Software,Introversion Software,"Indie, Strategy",...,"₩ 13,500",True,True,True,773.0,84.0,https://www.metacritic.com/game/pc/darwinia?ft...,NaN,https://shared.akamai.steamstatic.com/store_it...,http://www.darwinia.co.uk/
2,1510,Uplink,game,False,NaN,Uplink lets you play as a freelance hacker tac...,English,Introversion Software,Introversion Software,"Indie, Strategy",...,"₩ 13,500",True,True,True,1744.0,75.0,https://www.metacritic.com/game/pc/uplink-hack...,NaN,https://shared.akamai.steamstatic.com/store_it...,http://www.uplink.co.uk/


## 3. steamspy_indie_games

SteamSpy API로 수집한 인디게임 전체 데이터.

In [3]:
df_steamspy = pd.read_csv(RAW_DIR / 'steamspy_indie_games.csv')

print(f'steamspy_indie_games: {len(df_steamspy):,}행')
print(f'컬럼: {df_steamspy.columns.tolist()}')
df_steamspy.head(3)

steamspy_indie_games: 61,266행
컬럼: ['appid', 'spy_name', 'owners', 'positive', 'negative', 'price_spy', 'ccu', 'name_store', 'type', 'genres', 'release_date', 'developers']


,appid,spy_name,owners,positive,negative,price_spy,ccu,name_store,type,genres,release_date,developers
0,1002,Rag Doll Kung Fu,"20,000 .. 50,000",91,30,99,0,Rag Doll Kung Fu,game,['Indie'],"12 Oct, 2005",Mark Healey
1,1500,Darwinia,"0 .. 20,000",864,216,1199,2,Darwinia,game,"['Indie', 'Strategy']","1 Dec, 2005",Introversion Software
2,1510,Uplink,"500,000 .. 1,000,000",2143,216,1199,2,Uplink,game,"['Indie', 'Strategy']","23 Aug, 2006",Introversion Software


## 1. steam_indie_tags

SteamSpy API로 수집한 게임 태그 데이터.

In [4]:
df_tags = pd.read_csv(RAW_DIR / 'steam_indie_tags.csv')

print(f'steam_indie_tags: {len(df_tags):,}행')
print(f'컬럼: {df_tags.columns.tolist()}')
df_tags.head(3)

steam_indie_tags: 13,184행
컬럼: ['appid', 'name', 'developer', 'publisher', 'owners', 'positive', 'negative', 'price', 'tags']


,appid,name,developer,publisher,owners,positive,negative,price,tags
0,226620,Desktop Dungeons,QCF Design,QCF Design,"200,000 .. 500,000",1912,364,1499,"{""2D"": 36, ""RPG"": 103, ""Dwarf"": 18, ""Casual"": ..."
1,230210,ASYLUM,Senscape,Senscape,"0 .. 20,000",303,45,2499,"{""Dark"": 40, ""Gore"": 52, ""Indie"": 76, ""Gothic""..."
2,251570,7 Days to Die,The Fun Pimps,The Fun Pimps Entertainment LLC,"10,000,000 .. 20,000,000",327889,42157,4499,"{""FPS"": 3827, ""Voxel"": 4264, ""Action"": 3694, ""..."


## 데이터 로드 요약

In [5]:
print('=== 로드 완료 파일 목록 ===')
for f in sorted(RAW_DIR.glob('steam*.csv')):
    size_mb = f.stat().st_size / 1024 / 1024
    print(f'  {f.name:<45} {size_mb:6.1f} MB')

=== 로드 완료 파일 목록 ===
  steam_app_details.csv                           38.4 MB
  steam_indie_games_new.csv                       24.9 MB
  steam_indie_review_histogram.csv                38.6 MB
  steam_indie_review_summary.csv                   0.0 MB
  steam_indie_reviews.csv                          2.6 MB
  steam_indie_tags.csv                             5.4 MB
  steamspy_indie_games.csv                         8.4 MB


---

## steam_indie_games 전처리

`steamspy_indie_games`를 기반으로 두 분석 모집단을 선별한다.

- **초기 반응 그룹:** 리뷰 10개 이상
- **침묵 그룹:** 리뷰 0~9개

두 그룹 모두 같은 기간, EA/F2P/price=0, 부적합 장르, Indie 단독 제외 기준을 적용한다. 이후 `steam_app_details`, `steam_indie_tags` 데이터를 병합하여 최종 분석 데이터셋을 구성한다.

### 데이터 로드 및 파생 컬럼 생성

분석에 필요한 파생 컬럼을 생성한다.

- `total_reviews`: 긍정 + 부정 리뷰 합산
- `release_date`: datetime 변환
- `genres`: 문자열 → 리스트 파싱

In [6]:
df = df_steamspy.copy()

df['total_reviews'] = df['positive'] + df['negative']
df['release_date']  = pd.to_datetime(df['release_date'], errors='coerce')

def parse_genres(g):
    try:
        return ast.literal_eval(g)
    except Exception:
        return []

df['genres'] = df['genres'].apply(parse_genres)

print(f'원본: {len(df):,}개')
print(f'Early Access: {df["genres"].apply(lambda gl: "Early Access" in gl).sum():,}개 ({df["genres"].apply(lambda gl: "Early Access" in gl).mean():.1%})')
print(f'Free To Play: {df["genres"].apply(lambda gl: "Free To Play" in gl).sum():,}개 ({df["genres"].apply(lambda gl: "Free To Play" in gl).mean():.1%})')

원본: 61,266개
Early Access: 6,340개 (10.3%)
Free To Play: 3,446개 (5.6%)


### 분석 대상 필터링

다음 조건을 적용하여 두 그룹의 공통 모집단을 선별한다. 리뷰 수 기준 분기는 최종 저장 단계에서만 적용한다.

- 출시연도: 2023 ~ 2025년
- Early Access 제외
- Free To Play 장르 또는 `price_spy <= 0` 무료/가격 0 게임 제외
- 전체 장르 목록 기준 게임 수 중앙값(80개) 미만 장르를 포함한 게임 제외
  - 해당 장르: Nudity, Sexual Content, Education, Design & Illustration, Animation & Modeling, Game Development, Audio Production, Video Production, Software Training, Photo Editing, Web Publishing, Accounting, Movie, Short
- 추가 부적합 장르를 포함한 게임 제외: Massively Multiplayer, Violent, Gore, Utilities
- Indie 장르 단독 보유 게임 제외
- 성인 콘텐츠 태그 포함 게임 제외: Nudity, Sexual Content, Hentai, NSFW, Mature

Early Access와 F2P/무료 게임은 별도 데이터프레임(`df_ea`, `df_f2p`)으로 보존한다.

In [7]:
MIN_REVIEWS = 10

# 전체 장르 게임 수 중앙값(80) 미만 장르 + 추가 부적합 장르
EXCLUDE_GENRES = {
    # 중앙값 미만 (80개 미만)
    'Nudity', 'Sexual Content', 'Education', 'Design & Illustration',
    'Animation & Modeling', 'Game Development', 'Audio Production',
    'Video Production', 'Software Training', 'Photo Editing',
    'Web Publishing', 'Accounting', 'Movie', 'Short',
    # 추가 부적합 장르
    'Massively Multiplayer', 'Violent', 'Gore', 'Utilities',
}

# 장르가 아니라 태그로만 붙는 성인 콘텐츠도 분석 모집단에서 제외한다.
ADULT_TAGS = {'Nudity', 'Sexual Content', 'Hentai', 'NSFW', 'Mature'}

is_ea         = df['genres'].apply(lambda gl: 'Early Access' in gl)
is_f2p_genre  = df['genres'].apply(lambda gl: 'Free To Play' in gl)
is_zero_price = pd.to_numeric(df['price_spy'], errors='coerce').fillna(0) <= 0
is_f2p        = is_f2p_genre | is_zero_price
is_target_year = (df['release_date'].dt.year >= 2023) & (df['release_date'].dt.year <= 2025)

df_ea  = df[is_ea].copy()
df_f2p = df[~is_ea & is_f2p].copy()

# 공통 모집단: 리뷰 수 필터 없이 기간·EA·F2P/무료·장르 조건만 적용
df_f = df[is_target_year & (~is_ea) & (~is_f2p)].copy()

before = len(df_f)
df_f = df_f[~df_f['genres'].apply(lambda gl: bool(set(gl) & EXCLUDE_GENRES))].copy()
before_indie = len(df_f)
df_f = df_f[df_f['genres'].apply(lambda gl: gl != ['Indie'])].copy()

print(f'전체              : {len(df):,}개')
print(f'Early Access 제외 : {len(df_ea):,}개 → 별도 분석')
print(f'F2P/무료 제외     : {len(df_f2p):,}개 → 별도 분석')
print(f'  - Free To Play 장르: {is_f2p_genre.sum():,}개')
print(f'  - price=0 게임     : {is_zero_price.sum():,}개')
print(f'부적합 장르 제외  : {before - before_indie:,}개')
print(f'Indie 단독 제외   : {before_indie - len(df_f):,}개')
print(f'공통 모집단       : {len(df_f):,}개  (2023~2025년, EA·F2P/무료·부적합 장르·Indie 단독 제외, 리뷰 수 무관)')
print(f'  → 초기 반응 그룹 예상: {(df_f["total_reviews"] >= MIN_REVIEWS).sum():,}개 (리뷰 ≥{MIN_REVIEWS})')
print(f'  → 침묵 그룹 예상     : {(df_f["total_reviews"] < MIN_REVIEWS).sum():,}개 (리뷰 <{MIN_REVIEWS})')
print(f'\n출시연도 분포:')
print(df_f['release_date'].dt.year.value_counts().sort_index().to_string())

전체              : 61,266개
Early Access 제외 : 6,340개 → 별도 분석
F2P/무료 제외     : 6,543개 → 별도 분석
  - Free To Play 장르: 3,446개
  - price=0 게임     : 7,280개
부적합 장르 제외  : 231개
Indie 단독 제외   : 713개
공통 모집단       : 15,734개  (2023~2025년, EA·F2P/무료·부적합 장르·Indie 단독 제외, 리뷰 수 무관)
  → 초기 반응 그룹 예상: 8,997개 (리뷰 ≥10)
  → 침묵 그룹 예상     : 6,737개 (리뷰 <10)

출시연도 분포:
release_date
2023    5271
2024    7001
2025    3462


### 컬럼 정제

분석에 적합한 형태로 컬럼을 정리한다.

- `name`: `name_store` 우선, 없으면 `spy_name` 사용
- `price`: `price_spy` 컬럼명 변경
- `release_date`: 이후 병합 단계에서 SteamSpy 원본 날짜를 우선 보존
- 불필요 컬럼(`spy_name`, `name_store`, `type`) 제거

In [8]:
df_f['name'] = df_f['name_store'].fillna(df_f['spy_name'])
df_f = df_f.rename(columns={'price_spy': 'price'})
df_f['release_date'] = df_f['release_date'].dt.strftime('%Y-%m-%d')
df_f = df_f.drop(columns=['spy_name', 'name_store', 'type'])

print('전처리 완료')
print(df_f[['name', 'release_date', 'price']].head())

전처리 완료
                                   name release_date  price
564                    Desktop Dungeons   2023-04-18   1499
598                              ASYLUM   2025-03-13   2499
848                       7 Days to Die   2024-07-25   4499
868   Defender's Quest 2: Mists of Ruin   2025-01-30   1999
1178                 Secrets of Grindea   2024-02-29   1499


### steam_app_details 병합

- `name`, `developers`: `steam_app_details` 값을 우선 사용하고, 결측이면 SteamSpy 값을 유지
- `release_date`: SteamSpy 원본 날짜를 우선 보존하고, 결측일 때만 `steam_app_details` 날짜로 보완
- `genres`: SteamSpy의 리스트 형식을 유지 (`app_details`의 genres는 쉼표 문자열이라 제외)
- 나머지 `steam_app_details` 컬럼은 그대로 추가

> Store API의 `release_date`가 비어 있는 게임이 있어, Store 날짜로 무조건 대체하면 전처리 결과에 출시일 결측이 생길 수 있다.

In [9]:
common_cols_appdetails = sorted(set(df_f.columns) & set(df_app_details.columns))
print('games vs app_details 공통 컬럼:', common_cols_appdetails)

# genres는 steamspy 리스트 형식 유지 (app_details의 genres는 쉼표 문자열이라 ast.literal_eval 파싱 불가)
# release_date는 Store 결측이 많은 경우가 있어 SteamSpy 날짜를 우선 보존한다.
replace_cols2      = ['name', 'developers', 'release_date']
appdetails_replace = df_app_details[['appid'] + replace_cols2].copy()
appdetails_replace['release_date'] = pd.to_datetime(appdetails_replace['release_date'], errors='coerce')
appdetails_extra   = df_app_details.drop(columns=replace_cols2 + ['genres'])

df_f = df_f.merge(appdetails_replace, on='appid', how='left', suffixes=('_old', ''))
for col in ['name', 'developers']:
    df_f[col] = df_f[col].combine_first(df_f[f'{col}_old'])
    df_f = df_f.drop(columns=[f'{col}_old'])

# SteamSpy release_date(_old)를 우선 사용하고, 결측일 때만 Store release_date로 보완한다.
df_f['release_date'] = df_f['release_date_old'].combine_first(df_f['release_date'])
df_f = df_f.drop(columns=['release_date_old'])

df_f = df_f.merge(appdetails_extra, on='appid', how='left')

print('병합 결과 shape:', df_f.shape)
print('release_date 결측:', df_f['release_date'].isna().sum())
print('컬럼:', df_f.columns.tolist())

games vs app_details 공통 컬럼: ['appid', 'developers', 'genres', 'name', 'release_date']
병합 결과 shape: (15734, 34)
release_date 결측: 0
컬럼: ['appid', 'owners', 'positive', 'negative', 'price', 'ccu', 'genres', 'total_reviews', 'name', 'developers', 'release_date', 'type', 'is_free', 'controller_support', 'short_description', 'supported_languages', 'publishers', 'categories', 'coming_soon', 'currency', 'initial', 'final', 'discount_percent', 'initial_formatted', 'final_formatted', 'windows', 'mac', 'linux', 'recommendations_total', 'metacritic_score', 'metacritic_url', 'achievements_total', 'header_image', 'website']


### 타입 변환 및 컬럼 정리

- `price`: 정수(원 단위) → float (`/100`)
- `total_reviews`, `release_date`: numeric/datetime 변환
- `owners` 범위 문자열 → `owners_lower`(하한) + `owners_higher`(상한) 분리 후 `owners`, `ccu` 제거
- 분석에 불필요한 컬럼 15개 제거 (`type`, `is_free`, `controller_support`, `supported_languages`, `coming_soon`, `currency`, `initial`, `final`, `discount_percent`, `initial_formatted`, `final_formatted`, `metacritic_score`, `metacritic_url`, `header_image`, `website`)

In [10]:
df_f['price'] = pd.to_numeric(df_f['price'], errors='coerce') / 100
df_f['total_reviews'] = pd.to_numeric(df_f['total_reviews'], errors='coerce')
df_f['release_date'] = pd.to_datetime(df_f['release_date'], errors='coerce')

owners_clean = df_f['owners'].str.replace(',', '', regex=False)
df_f['owners_lower'] = pd.to_numeric(owners_clean.str.split(r'\.\.').str[0].str.strip(), errors='coerce')
df_f['owners_higher'] = pd.to_numeric(owners_clean.str.split(r'\.\.').str[1].str.strip(), errors='coerce')
df_f = df_f.drop(columns=['owners', 'ccu'])

drop_cols = [
    'type', 'is_free', 'controller_support', 'supported_languages',
    'coming_soon', 'currency', 'initial', 'final', 'discount_percent',
    'initial_formatted', 'final_formatted', 'metacritic_score',
    'metacritic_url', 'header_image', 'website'
]
df_f = df_f.drop(columns=[c for c in drop_cols if c in df_f.columns])

df_f['developers'] = df_f['developers'].fillna('unknown')
df_f['publishers'] = df_f['publishers'].fillna('unknown')

print(f'shape: {df_f.shape}')
print(f'컬럼: {df_f.columns.tolist()}')

shape: (15734, 19)
컬럼: ['appid', 'positive', 'negative', 'price', 'genres', 'total_reviews', 'name', 'developers', 'release_date', 'short_description', 'publishers', 'categories', 'windows', 'mac', 'linux', 'recommendations_total', 'achievements_total', 'owners_lower', 'owners_higher']


### steam_indie_tags 병합

`data/raw/steam_indie_tags.csv`에서 tags를 로드하여 공통 모집단(`df_f`)과 left join으로 병합한다.

- 태그는 리뷰 10개 이상 게임 위주로 수집되었으므로, 침묵 그룹(리뷰 0~9개)은 tags가 없을 수 있다.
- left join을 사용하여 침묵 그룹을 보존하고, tags 없는 게임은 NaN으로 유지한다.

태그 병합 이후에는 성인 콘텐츠 태그(`Nudity`, `Sexual Content`, `Hentai`, `NSFW`, `Mature`)를 가진 게임을 제외한다. 일부 게임은 `genres`에는 성인 장르가 없지만 `tags`에만 성인 콘텐츠 신호가 붙어 있으므로, 장르 필터와 별도로 처리한다.

In [11]:
df_tags = pd.read_csv(RAW_DIR / 'steam_indie_tags.csv')
df_tags['developer'] = df_tags['developer'].fillna('unknown')
df_tags['publisher'] = df_tags['publisher'].fillna('unknown')

# tags 병합 (left join: 침묵 그룹은 태그 미수집 게임 포함 가능)
tag_cols = [c for c in df_f.columns if 'tag' in c.lower()]
if tag_cols:
    df_f = df_f.drop(columns=tag_cols)

before = len(df_f)
df_f = df_f.merge(df_tags[['appid', 'tags']], on='appid', how='left')

tags_matched = df_f['tags'].notna().sum()
print(f'tags 병합 (left join): {before:,} → {len(df_f):,}행')
print(f'tags 보유: {tags_matched:,}개 / {len(df_f):,}개 ({tags_matched / len(df_f) * 100:.1f}%)')
print(f'  → 리뷰 ≥10 중 tags 없음: {df_f[df_f["total_reviews"] >= MIN_REVIEWS]["tags"].isna().sum():,}개')
print(f'  → 리뷰 <10  중 tags 없음: {df_f[df_f["total_reviews"] < MIN_REVIEWS]["tags"].isna().sum():,}개')
print(f'컬럼: {df_f.columns.tolist()}')

def parse_tag_names(value):
    if isinstance(value, dict):
        return list(value.keys())
    if isinstance(value, list):
        return value
    if pd.isna(value):
        return []
    if isinstance(value, str):
        for loader in (_json.loads, ast.literal_eval):
            try:
                parsed = loader(value)
                if isinstance(parsed, dict):
                    return list(parsed.keys())
                if isinstance(parsed, list):
                    return parsed
            except Exception:
                continue
    return []

# genres 필터로 잡히지 않는 성인 콘텐츠 태그 포함 게임 제외
before_adult_tag = len(df_f)
adult_tag_mask = df_f['tags'].apply(lambda value: bool(set(parse_tag_names(value)) & ADULT_TAGS))
adult_tag_removed = int(adult_tag_mask.sum())
df_f = df_f[~adult_tag_mask].copy()

print(f'성인 콘텐츠 태그 제외: {adult_tag_removed:,}개')
print(f'성인 태그 제외 후 공통 모집단: {len(df_f):,}개')
print(f'  → 초기 반응 그룹 예상: {(df_f["total_reviews"] >= MIN_REVIEWS).sum():,}개 (리뷰 ≥{MIN_REVIEWS})')
print(f'  → 침묵 그룹 예상     : {(df_f["total_reviews"] < MIN_REVIEWS).sum():,}개 (리뷰 <{MIN_REVIEWS})')


tags 병합 (left join): 15,734 → 15,734행
tags 보유: 13,021개 / 15,734개 (82.8%)
  → 리뷰 ≥10 중 tags 없음: 2,713개
  → 리뷰 <10  중 tags 없음: 0개
컬럼: ['appid', 'positive', 'negative', 'price', 'genres', 'total_reviews', 'name', 'developers', 'release_date', 'short_description', 'publishers', 'categories', 'windows', 'mac', 'linux', 'recommendations_total', 'achievements_total', 'owners_lower', 'owners_higher', 'tags']
성인 콘텐츠 태그 제외: 328개
성인 태그 제외 후 공통 모집단: 15,406개
  → 초기 반응 그룹 예상: 8,730개 (리뷰 ≥10)
  → 침묵 그룹 예상     : 6,676개 (리뷰 <10)


### 최종 저장

전처리가 완료된 공통 모집단을 `total_reviews`로 분기하여 두 파일로 저장한다.

- `steam_indie_games.csv` — 초기 반응 그룹 (리뷰 ≥10): 이후 모든 분석 노트북의 기본 데이터셋
- `steam_indie_games_silence.csv` — 침묵 그룹 (리뷰 0~9): 10번 노트북 비교 분석용

In [12]:
# 리뷰 수 기준 분기
df_response = df_f[df_f['total_reviews'] >= MIN_REVIEWS].copy()
df_silence  = df_f[df_f['total_reviews'] <  MIN_REVIEWS].copy()

# 저장
df_response.to_csv(PREPROCESSED_DIR / 'steam_indie_games.csv', index=False)
df_silence.to_csv(PREPROCESSED_DIR / 'steam_indie_games_silence.csv', index=False)

print(f'저장 완료 → steam_indie_games.csv         ({len(df_response):,}개, 리뷰 ≥{MIN_REVIEWS})')
print(f'저장 완료 → steam_indie_games_silence.csv ({len(df_silence):,}개, 리뷰 <{MIN_REVIEWS})')
print(f'컬럼: {df_response.columns.tolist()}')
print(f'\ntags 보유 (침묵): {df_silence["tags"].notna().sum():,}개 / {len(df_silence):,}개')

저장 완료 → steam_indie_games.csv         (8,730개, 리뷰 ≥10)
저장 완료 → steam_indie_games_silence.csv (6,676개, 리뷰 <10)
컬럼: ['appid', 'positive', 'negative', 'price', 'genres', 'total_reviews', 'name', 'developers', 'release_date', 'short_description', 'publishers', 'categories', 'windows', 'mac', 'linux', 'recommendations_total', 'achievements_total', 'owners_lower', 'owners_higher', 'tags']

tags 보유 (침묵): 6,676개 / 6,676개
